In [ ]:
#Qué hace: define y caracteriza la escalada antibiótica diaria como exposición dinámica.
#Clave: escalada = cambio de tratamiento, no juicio clínico.

In [18]:
# codigo base 
from google.cloud import bigquery
import pandas as pd
import numpy as np


PROJECT_ID = "mimic-pruebas"
HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU  = "physionet-data.mimiciv_3_1_icu"
DERIVED = "physionet-data.mimiciv_3_1_derived"

client = bigquery.Client(project=PROJECT_ID)

# === Params ===
WINDOWS_PATH = "05_ventanas_24h.parquet"
COHORT_PATH = "04_cohorte_base_T0.parquet"   # opcional pero recomendado
OUT_PATH = "13_abx_escalation_daily.parquet"

In [19]:
# 1 leer ventanas

df_win = pd.read_parquet(WINDOWS_PATH)

# Recomendado: filtrar hadm_id/subject_id para no bajar prescriptions de todo MIMIC
try:
    df_cohort = pd.read_parquet(COHORT_PATH)[["subject_id","hadm_id"]].drop_duplicates()
    df_keys = df_cohort
except Exception:
    df_keys = df_win[["subject_id","hadm_id"]].drop_duplicates()

df_win = df_win[["subject_id","hadm_id","icu_stay_id","day_idx","window_start","window_end"]].copy()

In [20]:
# 2 Extraer antibióticos desde prescriptions 

sql_abx = f"""
SELECT
  subject_id,
  hadm_id,
  starttime,
  stoptime,
  LOWER(drug) AS drug,
  route
FROM `{HOSP}.prescriptions`
WHERE LOWER(drug) LIKE '%cillin%'
   OR LOWER(drug) LIKE '%cef%'
   OR LOWER(drug) LIKE '%piperacillin%'
   OR LOWER(drug) LIKE '%tazobactam%'
   OR LOWER(drug) LIKE '%meropenem%'
   OR LOWER(drug) LIKE '%imipenem%'
   OR LOWER(drug) LIKE '%vancomycin%'
   OR LOWER(drug) LIKE '%linezolid%'
   OR LOWER(drug) LIKE '%colistin%'
   OR LOWER(drug) LIKE '%levoflox%'
"""
df_abx = client.query(sql_abx).to_dataframe()

# filtrar a la cohorte
df_abx = df_abx.merge(df_keys, on=["subject_id","hadm_id"], how="inner")

# normaliza
df_abx["drug_clean"] = (
    df_abx["drug"]
      .astype(str)
      .str.replace("-", " ", regex=False)
      .str.replace("/", " ", regex=False)
      .str.strip()
      .str.upper()
)

# jerarquia
antibiotic_group = {
    "AMOXICILLIN": 1,
    "CEFAZOLIN": 1,
    "PENICILLIN": 1,
    "AMPICILLIN": 1,

    "AMOXICILLIN CLAVULANATE": 2,
    "CEFUROXIME": 2,
    "CEFTRIAXONE": 2,
    "CEFOTAXIME": 2,

    "PIPERACILLIN TAZOBACTAM": 3,
    "CEFEPIME": 3,
    "LEVOFLOXACIN": 3,

    "MEROPENEM": 4,
    "IMIPENEM": 4,
    "CEFTAZIDIME AVIBACTAM": 4,

    "VANCOMYCIN": 5,
    "LINEZOLID": 5,
    "DAPTOMYCIN": 5,

    "COLISTIN": 6,
    "TIGECYCLINE": 6,
    "FOSFOMYCIN": 6,
}

df_abx["abx_rank"] = df_abx["drug_clean"].map(antibiotic_group).fillna(0).astype(int)

# gestion stoptime
df_abx["starttime"] = pd.to_datetime(df_abx["starttime"])
df_abx["stoptime"] = pd.to_datetime(df_abx["stoptime"])

# endtime robusto:
endtime = df_abx["stoptime"].copy()

# si stoptime es NaT o <= starttime, lo tratamos como evento mínimo (1 minuto)
mask_bad_end = endtime.isna() | (endtime <= df_abx["starttime"])
endtime.loc[mask_bad_end] = df_abx.loc[mask_bad_end, "starttime"] + pd.Timedelta(minutes=1)

df_abx["endtime_eff"] = endtime

E0000 00:00:1769605928.602294  952895 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [21]:
# 3 Expandir a “antibiótico activo por ventana” (solape)

# Merge ventanas x prescriptions por hadm/subject (reduce muchísimo)
df_m = df_win.merge(
    df_abx[["subject_id","hadm_id","starttime","endtime_eff","abx_rank","drug_clean"]],
    on=["subject_id","hadm_id"],
    how="left"
)

# Solape: [start,end) con [window_start, window_end)
overlap = (
    (df_m["starttime"] < df_m["window_end"]) &
    (df_m["endtime_eff"] > df_m["window_start"])
)
df_m = df_m[overlap].copy()

# agregamos máximo rank por día:
df_day = (
    df_m.groupby(["subject_id","hadm_id","icu_stay_id","day_idx"], as_index=False)
        .agg(
            abx_max_rank_day=("abx_rank","max"),
            abx_any_active_day=("abx_rank", lambda x: int((x > 0).any()))
        )
)

# completamos días sin antibiótico (rank 0):
df_day = df_win.merge(df_day, on=["subject_id","hadm_id","icu_stay_id","day_idx"], how="left")
df_day["abx_max_rank_day"] = df_day["abx_max_rank_day"].fillna(0).astype(int)
df_day["abx_any_active_day"] = df_day["abx_any_active_day"].fillna(0).astype(int)

In [22]:
# 4 Escalada = rank_day > máximo previo
df_day = df_day.sort_values(["icu_stay_id","day_idx"]).reset_index(drop=True)

# máximo previo hasta day-1
df_day["abx_max_rank_prev"] = (
    df_day.groupby("icu_stay_id")["abx_max_rank_day"]
          .apply(lambda s: s.shift(1).cummax())
          .reset_index(level=0, drop=True)
)
df_day["abx_max_rank_prev"] = df_day["abx_max_rank_prev"].fillna(0).astype(int)

df_day["abx_escalation_flag"] = (
    (df_day["day_idx"] > 0) &
    (df_day["abx_max_rank_day"] > df_day["abx_max_rank_prev"])
).astype(int)

In [23]:
# 5 Guardar output
out_cols = [
    "subject_id","hadm_id","icu_stay_id","day_idx",
    "abx_max_rank_day","abx_max_rank_prev","abx_escalation_flag","abx_any_active_day"
]
df_out = df_day[out_cols].copy()
df_out.to_parquet(OUT_PATH, index=False)

df_out.head()

,subject_id,hadm_id,icu_stay_id,day_idx,abx_max_rank_day,abx_max_rank_prev,abx_escalation_flag,abx_any_active_day
0,18421337,22413411,30000484,0,5,0,0,1
1,18421337,22413411,30000484,0,5,5,0,1
2,18421337,22413411,30000484,0,5,5,0,1
3,18421337,22413411,30000484,1,5,5,0,1
4,18421337,22413411,30000484,1,5,5,0,1


In [24]:
# % días con antibiótico activo
print("Pct days with any abx active:", df_out["abx_any_active_day"].mean())

# escaladas por día
print(df_out.groupby("day_idx")["abx_escalation_flag"].mean().head(10))

# sanity: 10 ejemplos con escalada
ex = df_out[df_out["abx_escalation_flag"]==1].head(10)
print(ex[["icu_stay_id","day_idx","abx_max_rank_prev","abx_max_rank_day"]])

Pct days with any abx active: 0.7373179080319474
day_idx
0    0.000000
1    0.010177
2    0.004761
3    0.003660
4    0.003124
5    0.002741
6    0.001900
7    0.001512
8    0.001178
9    0.001300
Name: abx_escalation_flag, dtype: float64
      icu_stay_id  day_idx  abx_max_rank_prev  abx_max_rank_day
155      30003598       19                  4                 5
643      30005000        4                  2                 5
683      30005707        1                  3                 5
1491     30016026        2                  1                 5
1577     30016265        2                  1                 5
3290     30030186        5                  3                 5
3448     30031755        4                  0                 2
3947     30038172        5                  2                 5
3982     30040446        1                  1                 2
3991     30041848        1                  3                 5
